# 01 — Data exploration

This notebook gives a first look at the three data streams that flow into the cardiometabolic graph:

1. **MIMIC-IV demo** — clinical labs and vitals.
2. **NHANES 2017-18** — behavioral/lifestyle variables.
3. **Synthetic engagement events** — app opens, message responses, glucose logs.

Goal: characterize each stream and identify the columns that will become graph nodes.

> Prereqs: run `make synth` and (optionally) drop MIMIC + NHANES into `data/raw/`. See `data/raw/README` for layout.

In [ ]:
import pandas as pd
from pathlib import Path
from etl._common import synthetic_path, raw_path

events = pd.read_parquet(synthetic_path() / 'engagement_events.parquet')
archetypes = pd.read_parquet(synthetic_path() / 'engagement_archetypes.parquet')
print('events:', events.shape, 'patients:', events.patient_id.nunique())
events.head()

In [ ]:
events.groupby('kind')['value'].agg(['count', 'mean'])

In [ ]:
by_arch = events.merge(archetypes, on='patient_id').groupby(['archetype', 'kind']).size().unstack(fill_value=0)
by_arch

Expect *power_user* to dominate `app_open` counts, *early_dropout* to skew low, *episodic* to spike around lab dates. If the data doesn't match the documented mechanism, regenerate with a different seed and investigate.